# Week 49, Day 5: Cognitive maps and the successor representation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w049/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Skew against the direction of travel

Build T for a 50-state track where the agent steps right with probability p_right and left otherwise (staying put at the ends). Compute M = (I - gamma T)^-1 with gamma = 0.9, take the column for state 25 as that cell's field, and find its centre of mass for p_right of 0.5 and 0.9.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2017)
n, gamma = 50, 0.9

def track_T(p_right):
    T = np.zeros((n, n))
    for s in range(n):
        T[s, min(s + 1, n - 1)] += p_right; T[s, max(s - 1, 0)] += 1 - p_right
    return T

com = {}
for p in [0.5, 0.9]:
    M = ...
    field = M[:, 25]
    com[p] = (field * np.arange(n)).sum() / field.sum()
    plt.plot(field / field.max(), label=f"p_right = {p}")
    print(f"p_right = {p}: field peaks at {np.argmax(field)}, centre of mass {com[p]:.2f}")
plt.axvline(25, color="k", ls=":"); plt.xlabel("position"); plt.ylabel("field (normalised)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_symmetric():
    "with an unbiased random walk, the field's centre of mass is at the cell's location"
    assert abs(com[0.5] - 25) < 0.5
def test_skew():
    "with a rightward policy, the field's mass shifts at least 4 states to the left"
    assert com[0.9] < 21

test_symmetric()
test_skew()
print("All checks passed")

### Exercise: An SR learned from running

Let the agent run 40,000 steps with the rightward policy, returning to state 0 when it reaches the end, and update M with the TD rule (alpha = 0.1). Compare the learned field of state 25 with the true one.

In [ ]:
M_true = np.linalg.inv(np.eye(n) - gamma * track_T(0.9))
M_td = np.zeros((n, n)); I = np.eye(n); s = 0
moves_right = r.random(40000) < 0.9
for t in range(40000):
    s2 = min(s + 1, n - 1) if moves_right[t] else max(s - 1, 0)
    ...
    s = 0 if s2 == n - 1 else s2
print(f"correlation between learned and true field of state 25: {np.corrcoef(M_td[:, 25], M_true[:, 25])[0, 1]:.3f}")
plt.plot(M_true[:, 25], label="true"); plt.plot(M_td[:, 25], label="learned"); plt.legend(); plt.xlabel("position"); plt.show()

In [ ]:
import numpy as np
def test_field():
    "the learned field of state 25 correlates above 0.95 with the true one"
    assert np.corrcoef(M_td[:, 25], M_true[:, 25])[0, 1] > 0.95

test_field()
print("All checks passed")

### Exercise: Periodic eigenvectors of a predictive map

Build a random-walk transition matrix for a 12 by 12 room (moves to the four neighbours inside the room), compute its SR with gamma 0.95 and the eigenvectors sorted by eigenvalue. For eigenvectors 1, 3, 10 and 20, count sign changes between neighbouring cells along all rows and columns.

In [ ]:
k = 12; N = k * k; A = np.zeros((N, N))
for i in range(k):
    for j in range(k):
        for di, dj in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            a, b = i + di, j + dj
            if 0 <= a < k and 0 <= b < k:
                A[i * k + j, a * k + b] = 1
T2 = A / A.sum(1, keepdims=True)
M2 = ...
w, v = np.linalg.eig(M2); order = np.argsort(-w.real); v = v[:, order].real

def sign_changes(e):
    e = np.sign(e.reshape(k, k))
    return int(np.sum(np.diff(e, axis=0) != 0) + np.sum(np.diff(e, axis=1) != 0))

idx = [1, 3, 10, 20]
changes = [sign_changes(v[:, q]) for q in idx]
print(dict(zip(idx, changes)), "sign changes by eigenvector index")
fig, ax = plt.subplots(1, 4, figsize=(9, 2.4))
for a_, q in zip(ax, idx):
    a_.imshow(v[:, q].reshape(k, k), cmap="RdBu"); a_.set_title(f"eigenvector {q}"); a_.axis("off")
plt.show()

In [ ]:
import numpy as np
def test_increasing():
    "eigenvectors further down the spectrum change sign more often across the room"
    assert changes[0] < changes[2] < changes[3]
def test_smooth():
    "the first non-constant eigenvector changes sign fewer than 30 times across all rows and columns"
    assert changes[0] < 30

test_increasing()
test_smooth()
print("All checks passed")